# Function 7 Analysis - Week 8

**Function description:** You're tasked with optimising an ML model by tuning six hyperparameters (e.g., learning rate, regularisation strength, number of hidden layers). The objective is to maximise the model's performance score (accuracy/F1) while treating the relationship between inputs and output as a black-box function. Because this is a common model, prior best-practice guidance can inform the initial search space.

**New datapoint (Week 7):** `(0.0000, 0.0000, 0.0000, 0.3092, 0.3788, 0.6977)` returned **≈1.7220**, slightly below last week’s **1.7805** at `(0.0000, 0.0000, 0.0000, 0.2438, 0.3600, 0.6771)`. Total datapoints: **37**.

**Analysis:** The higher x4 and mid x6 still sit on the productive ridge, but the drop suggests the sweet spot remains closer to x4≈0.24–0.27 and x6≈0.67–0.72 rather than pushing x4 higher.

**Recommendation for this week’s BO changes:** Keep EI with `xi` 0.02–0.03, cap `x6 < 0.75` (bias 0.65–0.72), and maintain a soft floor for `x4` around 0.20–0.27. Add a small diversity/repulsion term so candidates spread around the `(x4, x6)` neighbourhood instead of collapsing to a single corner, while keeping `x1/x2/x3` very low and `x5` mid (~0.34–0.38).


## Loading and Displaying the Data

We load the inputs and outputs for function 7. Week 7’s EI pick `(0.0, 0.0, 0.0, 0.3092, 0.3788, 0.6977)` returned **≈1.7220** (slightly below best). Week 6 `(0.0, 0.0, 0.0, 0.2438, 0.36, 0.6771)` remains best at ≈1.7805. Total datapoints: **37**. Week 4’s `(0.0, 0.0804, 0.0, 0.0543, 0.3607, 0.7677)` was ≈1.025 (historical).


In [1]:
from pathlib import Path
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="ticks", context="notebook")
path = Path("../../initial_data/function_7")
X = np.load(path / "initial_inputs.npy")
y = np.load(path / "initial_outputs.npy")

# Week 1–7 new points
X_new_point_week_1 = np.array([[0.800000, 0.800000, 0.800000, 0.830000, 0.450000, 0.700000]])
y_new_point_week_1 = np.array([0.0344995016351187])
X_new_point_week_2 = np.array([[0.100000, 0.100000, 0.950000, 0.200000, 0.360000, 0.800000]])
y_new_point_week_2 = np.array([1.3138004996124066])
X_new_point_week_3 = np.array([[0.000000, 0.074100, 0.000000, 0.197300, 0.379200, 0.727100]])
y_new_point_week_3 = np.array([1.6455342546819547])
X_new_point_week_4 = np.array([[0.000000, 0.080400, 0.000000, 0.054300, 0.360700, 0.767700]])
y_new_point_week_4 = np.array([1.024932491434584])
X_new_point_week_5 = np.array([[0.000000, 0.000000, 0.000000, 0.233700, 0.366100, 1.000000]])
y_new_point_week_5 = np.array([0.7102053800176464])
X_new_point_week_6 = np.array([[0.000000, 0.000000, 0.000000, 0.243800, 0.360000, 0.677100]])
y_new_point_week_6 = np.array([1.780488134201001])
X_new_point_week_7 = np.array([[0.000000, 0.000000, 0.000000, 0.309200, 0.378800, 0.697700]])
y_new_point_week_7 = np.array([1.7220])

X = np.vstack([
    X,
    X_new_point_week_1,
    X_new_point_week_2,
    X_new_point_week_3,
    X_new_point_week_4,
    X_new_point_week_5,
    X_new_point_week_6,
    X_new_point_week_7,
])
y = np.concatenate([
    y,
    y_new_point_week_1,
    y_new_point_week_2,
    y_new_point_week_3,
    y_new_point_week_4,
    y_new_point_week_5,
    y_new_point_week_6,
    y_new_point_week_7,
])

df = pd.DataFrame(X, columns=["x1", "x2", "x3", "x4", "x5", "x6"]); df["y"] = y
display(df)
print("df sorted by y")
df_sorted = df.sort_values("y", ascending=False).reset_index(drop=True)
df_sorted["x_avg"] = df_sorted[["x1", "x2", "x3", "x4", "x5", "x6"]].mean(axis=1)
display(df_sorted)


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


df sorted by y


,x1,x2,x3,x4,x5,x6,y,x_avg
0,0.000000,0.000000,0.000000,0.243800,0.360000,0.677100,1.780488,0.213483
1,0.000000,0.000000,0.000000,0.309200,0.378800,0.697700,1.722000,0.230950
2,0.000000,0.074100,0.000000,0.197300,0.379200,0.727100,1.645534,0.229617
3,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968,0.361084
4,0.100000,0.100000,0.950000,0.200000,0.360000,0.800000,1.313800,0.418333
5,0.000000,0.080400,0.000000,0.054300,0.360700,0.767700,1.024932,0.210517
6,0.000000,0.000000,0.000000,0.233700,0.366100,1.000000,0.710205,0.266633
7,0.881647,0.204450,0.414474,0.420385,0.264915,0.730660,0.675142,0.486089
8,0.148647,0.033943,0.728806,0.316066,0.021769,0.516918,0.611526,0.294358
9,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433,0.546184


### Week 7 Outcome and Current Best

Week 3’s EI suggestion `(0.0, 0.0741, 0.0, 0.1973, 0.3792, 0.7271)` delivered **≈1.65** (former best). Week 4 `(0.0, 0.0804, 0.0, 0.0543, 0.3607, 0.7677)` was **≈1.025**. Week 7 evaluation `(0.0000, 0.0000, 0.0000, 0.3092, 0.3788, 0.6977)` returned **≈1.7220** (slightly below best). Week 6 `(0.2438, 0.36, 0.6771)` remains current best at ≈1.7805.


- **Week 7 (new):** `(0.0000, 0.0000, 0.0000, 0.3092, 0.3788, 0.6977)` yielded **≈1.7220** (slightly below best). Current best remains Week 6: `(0.2438, 0.36, 0.6771)` ≈1.7805.
- Recommendation for next BO step: keep EI with `xi` 0.02–0.03, **cap x6 < 0.75** (bias 0.65–0.72), soft floor x4 ~0.20–0.27, and diversity/repulsion on (x4, x6) so candidates spread around the sweet spot.


In [2]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize
np.random.seed(42)
# Per-feature lengthscales with bounds (assuming little noise, no WhiteKernel)
kernel = Matern(
    length_scale=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    length_scale_bounds=(0.2, 5.0),
    nu=2.5
)
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=5)
gp.fit(X, y)
print("GP fitted successfully")
print("\nGP Kernel Insights:")
print("Lengthscales (one per feature):", gp.kernel_.length_scale)
print("Full kernel parameters:", gp.kernel_.get_params())


GP fitted successfully

GP Kernel Insights:
Lengthscales (one per feature): [1.36805372 1.91627482 5.         0.39748116 0.3436851  0.40450738]
Full kernel parameters: {'length_scale': array([1.36805372, 1.91627482, 5.        , 0.39748116, 0.3436851 ,
       0.40450738]), 'length_scale_bounds': (0.2, 5.0), 'nu': 2.5}


d:\OneDrive\Documents\cursor\imperial_college_capstone\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter length_scale is close to the specified upper bound 5.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


## Finding the Next Point to Evaluate (Week 8 config)

We keep EI with `xi` **0.025** (0.02–0.03), cap `x6 < 0.75` (bias 0.65–0.72), and a soft floor for `x4` around **0.20–0.27** (sweet spot closer to x4≈0.24–0.27 after Week 7). Restarts bias `x1/x2/x3` near zero and `x5` mid (≈0.34–0.38). A diversity/repulsion term on `(x4, x6)` spreads candidates around the neighbourhood instead of collapsing to one corner.


In [3]:
from scipy.stats import norm

xi = 0.025  # light exploration while staying on the ridge
repulsion_weight = 0.03  # slightly stronger to spread candidates after Week 7
repulsion_bandwidth = 0.04
repulsion_dims = (3, 5)  # x4, x6
bounds = [
    (0.0, 0.12),  # x1 kept low
    (0.0, 0.12),  # x2 kept low
    (0.0, 0.12),  # x3 kept low
    (0.20, 0.28), # x4 sweet spot ~0.24–0.27 (tightened after Week 7)
    (0.30, 0.50), # x5 mid-band
    (0.55, 0.75), # x6 capped below 0.75, biased near 0.65–0.72
]

def clip_to_bounds(x, bounds):
    x = np.array(x, dtype=float)
    for i, (lo, hi) in enumerate(bounds):
        x[i] = np.clip(x[i], lo, hi)
    return x


def diversity_penalty(x, X_seen, dims=repulsion_dims, bandwidth=repulsion_bandwidth):
    if X_seen is None or len(X_seen) == 0:
        return 0.0
    diffs = X_seen[:, list(dims)] - x[list(dims)]
    dists = np.linalg.norm(diffs, axis=1)
    weights = np.exp(-(dists ** 2) / (2 * bandwidth ** 2))
    return weights.mean()


# Expected Improvement acquisition function with a small repulsion term
# on (x4, x6) to spread candidates near the capped region.
def expected_improvement(x, gp, y_best, xi=xi, X_seen=None):
    x = clip_to_bounds(x, bounds)
    mu, sigma = gp.predict(x.reshape(1, -1), return_std=True)

    sigma = sigma + 1e-9
    improvement = mu - y_best - xi
    Z = improvement / sigma
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

    penalty = diversity_penalty(x, X_seen, dims=repulsion_dims, bandwidth=repulsion_bandwidth)
    adjusted_ei = ei[0] - repulsion_weight * penalty

    return -adjusted_ei  # negative for minimization


def biased_restart():
    return np.array([
        np.random.uniform(0.0, 0.08),   # x1
        np.random.uniform(0.0, 0.08),   # x2
        np.random.uniform(0.0, 0.08),   # x3
        np.random.uniform(0.22, 0.28),  # x4 sweet spot ~0.24–0.27
        np.random.uniform(0.34, 0.38),  # x5 mid
        np.random.uniform(0.65, 0.72),  # x6 below cap
    ])


def global_restart():
    return np.array([np.random.uniform(lo, hi) for lo, hi in bounds])


# Get current best
y_best = y.max()
best_idx = y.argmax()
print(f"Current best score: {y_best:.4f}")
print(f"Current best point: {X[best_idx]}")

# Optimize acquisition function with multiple random restarts
n_restarts = 40
best_acquisition = np.inf
best_candidate = None

np.random.seed(42)
for i in range(n_restarts):
    # Bias most restarts to the recommended region; keep a few global restarts
    x0 = biased_restart() if i < int(0.7 * n_restarts) else global_restart()
    x0 = clip_to_bounds(x0, bounds)
    result = minimize(
        lambda x: expected_improvement(x, gp, y_best, xi=xi, X_seen=X),
        x0=x0,
        bounds=bounds,
        method='L-BFGS-B'
    )

    if result.fun < best_acquisition:
        best_acquisition = result.fun
        best_candidate = result.x

next_point = clip_to_bounds(best_candidate, bounds)
mu_pred, sigma_pred = gp.predict(next_point.reshape(1, -1), return_std=True)

print(f"\n{'='*60}")
print("BAYESIAN OPTIMIZATION RECOMMENDATION (Expected Improvement)")
print(f"{'='*60}")
print(f"\nNext point to evaluate:")
print(f"  x1={next_point[0]:.4f}, x2={next_point[1]:.4f}, x3={next_point[2]:.4f}")
print(f"  x4={next_point[3]:.4f}, x5={next_point[4]:.4f}, x6={next_point[5]:.4f}")
print(f"\nPredicted output: {mu_pred[0]:.4f} ± {sigma_pred[0]:.4f}")
print(f"Expected Improvement (penalized): {-best_acquisition:.6f}")


Current best score: 1.7805
Current best point: [0.     0.     0.     0.2438 0.36   0.6771]

BAYESIAN OPTIMIZATION RECOMMENDATION (Expected Improvement)

Next point to evaluate:
  x1=0.0000, x2=0.0000, x3=0.1200
  x4=0.2433, x5=0.4002, x6=0.6110

Predicted output: 1.7019 ± 0.0939
Expected Improvement (penalized): 0.005290


<!-- Distance analysis removed per latest guidance. -->


## Summary and recommended point
- **Current best:** `0.000000-0.000000-0.000000-0.243800-0.360000-0.677100` (≈1.7805, Week 6).
- Week 7 evaluated: `0.000000-0.000000-0.000000-0.309200-0.378800-0.697700` → ≈1.7220 (slightly below best).
- **Recommended next point (format: 6 decimals, dash-separated):** See the output of the BO cell above for this week's value.
- Best alternatives (recap):
  - Week 3: `0.000000-0.074100-0.000000-0.197300-0.379200-0.727100` (≈1.6455)
  - Week 2: `0.100000-0.100000-0.950000-0.200000-0.360000-0.800000` (≈1.3138)

### What changed and why
After Week 7 (1.7220, slightly below best): we keep EI with **xi=0.025**, tighten **x4** to 0.20–0.28 (sweet spot ~0.24–0.27), cap **x6 < 0.75**, and increase the diversity/repulsion weight slightly so candidates spread around the (x4, x6) neighbourhood instead of collapsing to one corner. Restarts bias x1/x2/x3 near zero and x5 mid (~0.34–0.38).

